# Density Overlap Integral Between Two Molecules

The **density overlap integral** measures how much the electron densities of two molecules co-occupy the same region of space:

$$\Omega_{AB} = \int \rho_A(\mathbf{r})\, \rho_B(\mathbf{r})\, d\mathbf{r}$$

Expanding each density in atomic orbital (AO) basis functions $\phi_\mu$:

$$\rho(\mathbf{r}) = \sum_{\mu\nu} P_{\mu\nu}\, \phi_\mu(\mathbf{r})\, \phi_\nu(\mathbf{r})$$

the integral becomes a contraction over density matrix elements and **4-center 1-electron overlap integrals**:

$$\Omega_{AB} = \sum_{\mu\nu \in A}\; \sum_{\lambda\sigma \in B} P^A_{\mu\nu}\, P^B_{\lambda\sigma} \int \phi^A_\mu(\mathbf{r})\, \phi^A_\nu(\mathbf{r})\, \phi^B_\lambda(\mathbf{r})\, \phi^B_\sigma(\mathbf{r})\, d\mathbf{r}$$

This notebook covers:
1. Building density matrices from **canonical MO (CMO)** coefficients
2. Building density matrices from **localized MO (LMO)** coefficients (Boys & Pipek–Mezey)
3. **Numerical grid integration** of $\Omega_{AB}$
4. **Approximate analytic formula** using the AO overlap matrix
5. **Distance dependence** of $\Omega_{AB}$
6. **LMO pair decomposition** — which orbital pairs drive the overlap

## 0. Imports

In [ ]:
# Uncomment to install if needed
# !pip install pyscf numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, lo
from pyscf.dft import gen_grid, numint

print('PySCF and NumPy loaded.')

## 1. Define two molecules

Wat1 and Wat2 — two water molecules at their specified geometries, STO-3G basis.

In [ ]:
molA = gto.M(
    atom='''
        O  -1.464000   0.099000   0.300000
        H  -1.956000   0.624000  -0.340000
        H  -1.797000  -0.799000   0.206000
    ''',
    basis='sto-3g',
    verbose=0
)

molB = gto.M(
    atom='''
        O   1.369000   0.146000  -0.395000
        H   1.894000   0.486000   0.335000
        H   0.451000   0.165000  -0.083000
    ''',
    basis='sto-3g',
    verbose=0
)

print(f'Wat1: {molA.nao_nr()} AOs,  {molA.nelectron} electrons')
print(f'Wat2: {molB.nao_nr()} AOs,  {molB.nelectron} electrons')

## 2. Run RHF

In [ ]:
mfA = scf.RHF(molA); mfA.verbose = 0; mfA.run()
mfB = scf.RHF(molB); mfB.verbose = 0; mfB.run()

print(f'E(A) = {mfA.e_tot:.6f} Eh')
print(f'E(B) = {mfB.e_tot:.6f} Eh')

## 3. Density matrix from canonical MO (CMO) coefficients

The 1-particle reduced density matrix is:

$$\mathbf{P} = 2\,\mathbf{C}_{\text{occ}}\,\mathbf{C}_{\text{occ}}^T$$

where $\mathbf{C}_{\text{occ}}$ contains only the **occupied** MO columns (shape `nAO × n_occ`).
The factor of 2 accounts for double-occupation of each spatial orbital in closed-shell RHF.

`mf.make_rdm1()` does exactly this internally — we verify they agree.

In [ ]:
def dm_from_cmo(mf):
    """Build density matrix from occupied canonical MO coefficients."""
    C     = mf.mo_coeff              # (nAO, nMO)
    occ   = mf.mo_occ                # occupation numbers: 2.0 or 0.0
    C_occ = C[:, occ > 0]            # (nAO, n_occ)  — keep occupied columns
    return 2 * C_occ @ C_occ.T       # (nAO, nAO)


dmA_cmo = dm_from_cmo(mfA)
dmB_cmo = dm_from_cmo(mfB)

# Verify against PySCF's built-in
print('Max |P_CMO - make_rdm1| for A:', np.max(np.abs(dmA_cmo - mfA.make_rdm1())))
print('Max |P_CMO - make_rdm1| for B:', np.max(np.abs(dmB_cmo - mfB.make_rdm1())))

# Sanity: Tr[PS] = N_electrons  (S = AO overlap)
SA = molA.intor('int1e_ovlp')
SB = molB.intor('int1e_ovlp')
print(f'Tr[P_A S_A] = {np.trace(dmA_cmo @ SA):.4f}  (should be {molA.nelectron})')
print(f'Tr[P_B S_B] = {np.trace(dmB_cmo @ SB):.4f}  (should be {molB.nelectron})')

## 4. Density matrix from localized MO (LMO) coefficients

LMOs are a **unitary rotation** of the occupied CMOs:

$$\mathbf{C}_{\text{LMO}} = \mathbf{C}_{\text{occ}}\,\mathbf{U}, \qquad \mathbf{U}\mathbf{U}^T = \mathbf{I}$$

Because $\mathbf{U}$ is unitary, the density matrix is **invariant**:

$$\mathbf{P} = 2\,\mathbf{C}_{\text{LMO}}\mathbf{C}_{\text{LMO}}^T
             = 2\,\mathbf{C}_{\text{occ}}\underbrace{\mathbf{U}\mathbf{U}^T}_{=\mathbf{I}}\mathbf{C}_{\text{occ}}^T
             = 2\,\mathbf{C}_{\text{occ}}\mathbf{C}_{\text{occ}}^T$$

The density is a physical observable — it cannot depend on the orbital representation.

We demonstrate two common localization schemes:
- **Boys** (Foster–Boys): minimizes spread $\langle r^2\rangle - \langle r\rangle^2$ of each orbital
- **Pipek–Mezey (PM)**: maximizes atomic charge localization (preserves σ/π separation)

In [ ]:
def dm_from_lmo(mf, mol, method='boys'):
    """
    Localize occupied MOs and return (C_lmo, P).
    method: 'boys' or 'pm'
    """
    C_occ = mf.mo_coeff[:, mf.mo_occ > 0]   # (nAO, n_occ)

    if method == 'boys':
        loc = lo.Boys(mol, C_occ)
    elif method == 'pm':
        loc = lo.PM(mol, C_occ)
    else:
        raise ValueError(f'Unknown method: {method}')

    C_lmo = loc.kernel()                     # (nAO, n_occ)
    P     = 2 * C_lmo @ C_lmo.T             # (nAO, nAO)
    return C_lmo, P


C_boysA, dmA_boys = dm_from_lmo(mfA, molA, 'boys')
C_boysB, dmB_boys = dm_from_lmo(mfB, molB, 'boys')

C_pmA, dmA_pm = dm_from_lmo(mfA, molA, 'pm')
C_pmB, dmB_pm = dm_from_lmo(mfB, molB, 'pm')

print('Molecule A — max |P_CMO - P_Boys|:', np.max(np.abs(dmA_cmo - dmA_boys)))
print('Molecule A — max |P_CMO - P_PM|:  ', np.max(np.abs(dmA_cmo - dmA_pm)))
print('Molecule B — max |P_CMO - P_Boys|:', np.max(np.abs(dmB_cmo - dmB_boys)))
print('Molecule B — max |P_CMO - P_PM|:  ', np.max(np.abs(dmB_cmo - dmB_pm)))
print('\n→ All density matrices are numerically identical (differences are floating-point noise).')

### LMO orbital centroids

Unlike delocalized CMOs, each LMO has a well-defined centre — useful for
assigning orbitals to bonds, lone pairs, or functional groups.

In [ ]:
def lmo_centroids(mol, C_lmo):
    """Return (n_occ, 3) array of LMO expectation values <r>."""
    r_ints = mol.intor('int1e_r')           # (3, nAO, nAO) dipole integrals
    return np.array([
        [np.einsum('ij,ji->', r_ints[k], np.outer(C_lmo[:, i], C_lmo[:, i]))
         for k in range(3)]
        for i in range(C_lmo.shape[1])
    ])                                      # (n_occ, 3)


cents_A = lmo_centroids(molA, C_boysA)
print('Boys LMO centroids for molecule A (Bohr):')
print(f'  {"LMO":>4}  {"x":>8}  {"y":>8}  {"z":>8}')
for i, c in enumerate(cents_A):
    print(f'  {i:>4}  {c[0]:>8.3f}  {c[1]:>8.3f}  {c[2]:>8.3f}')

## 5. Numerical grid integration of $\Omega_{AB}$

$$\Omega_{AB} \approx \sum_g w_g\, \rho_A(\mathbf{r}_g)\, \rho_B(\mathbf{r}_g)$$

Because the density matrix is the same regardless of orbital basis (CMO or LMO),
the total $\Omega_{AB}$ is identical in all cases — verified below.

In [ ]:
def build_union_grid(mol1, mol2, level=3):
    g1 = gen_grid.Grids(mol1); g1.level = level; g1.build()
    g2 = gen_grid.Grids(mol2); g2.level = level; g2.build()
    return np.vstack([g1.coords, g2.coords]), np.concatenate([g1.weights, g2.weights])


def eval_density(mol, dm, coords):
    ao  = numint.eval_ao(mol, coords)              # (npts, nAO)
    return np.einsum('pi,ij,pj->p', ao, dm, ao)   # (npts,)


def density_overlap(mol1, dm1, mol2, dm2, level=3):
    coords, weights = build_union_grid(mol1, mol2, level)
    rho1 = eval_density(mol1, dm1, coords)
    rho2 = eval_density(mol2, dm2, coords)
    return float(np.dot(weights, rho1 * rho2))


coords, weights = build_union_grid(molA, molB, level=3)
rhoA = eval_density(molA, dmA_cmo, coords)
rhoB = eval_density(molB, dmB_cmo, coords)

omega_cmo   = np.dot(weights, rhoA * rhoB)
omega_boys  = density_overlap(molA, dmA_boys, molB, dmB_boys)
omega_pm    = density_overlap(molA, dmA_pm,   molB, dmB_pm)

print(f'Ω_AB  (CMO density matrix)  = {omega_cmo:.8f}')
print(f'Ω_AB  (Boys density matrix) = {omega_boys:.8f}')
print(f'Ω_AB  (PM density matrix)   = {omega_pm:.8f}')
print('\n→ All three are identical — the total overlap is basis-representation-invariant.')

## 6. Approximate analytic formula

Using the cross-block AO overlap matrix $S^{AB}_{\mu\lambda} = \langle\phi^A_\mu|\phi^B_\lambda\rangle$:

$$\Omega_{AB} \approx \operatorname{Tr}\!\left[P_A\, S^{AB}\, P_B\, (S^{AB})^T\right]$$

In [ ]:
def build_cross_overlap(mol1, mol2):
    nA = mol1.nao_nr()
    molAB = gto.M(
        atom=mol1._atom + mol2._atom,
        basis=mol1.basis,
        verbose=0
    )
    S_full = molAB.intor('int1e_ovlp')
    return S_full[:nA, nA:]


S_AB = build_cross_overlap(molA, molB)
print(f'Cross-overlap S_AB shape: {S_AB.shape}')
print(f'Max |S_AB|: {np.max(np.abs(S_AB)):.2e}  (small → molecules well-separated)')

omega_analytic = np.einsum('ij,jk,kl,li->', dmA_cmo, S_AB, dmB_cmo, S_AB.T)
print(f'\nΩ_AB (analytic approx) = {omega_analytic:.8f}')
print(f'Ω_AB (numerical)       = {omega_cmo:.8f}')
print(f'Difference:              {abs(omega_cmo - omega_analytic):.2e}')

## 7. Distance dependence

In [ ]:
distances_angstrom = np.linspace(2.0, 8.0, 13)
omegas = []

for d in distances_angstrom:
    mol_b = gto.M(
        atom=f'O {d:.3f} 0 0; H {d:.3f} 0.757 0.586; H {d:.3f} -0.757 0.586',
        basis='sto-3g', verbose=0
    )
    mf_b = scf.RHF(mol_b); mf_b.verbose = 0; mf_b.run()
    omegas.append(density_overlap(molA, dmA_cmo, mol_b, mf_b.make_rdm1()))
    print(f'd = {d:.1f} Å  →  Ω = {omegas[-1]:.4e}')

plt.figure(figsize=(7, 4))
plt.semilogy(distances_angstrom, omegas, 'o-')
plt.xlabel('O–O distance (Å)')
plt.ylabel(r'Density overlap $\Omega_{AB}$  (log scale)')
plt.title('Density overlap vs. intermolecular separation (H₂O · · · H₂O, STO-3G)')
plt.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('density_overlap_vs_distance.png', dpi=150)
plt.show()

## 8. LMO pair decomposition of $\Omega_{AB}$

Even though the *total* $\Omega_{AB}$ is the same for CMOs and LMOs, LMOs give a physically
meaningful **per-orbital-pair decomposition** because each LMO is spatially local.

Using the LMO densities:

$$\rho_A(\mathbf{r}) = 2\sum_i |\varphi_i^A(\mathbf{r})|^2, \qquad
  \rho_B(\mathbf{r}) = 2\sum_j |\varphi_j^B(\mathbf{r})|^2$$

the overlap decomposes exactly into pair contributions:

$$\Omega_{AB} = \sum_{i \in A}\sum_{j \in B} \omega_{ij}, \qquad
  \omega_{ij} = 4\int |\varphi_i^A(\mathbf{r})|^2 |\varphi_j^B(\mathbf{r})|^2\, d\mathbf{r}$$

Each $\omega_{ij}$ is a **non-negative** number representing how much LMO $i$ on molecule A
overlaps with LMO $j$ on molecule B. Localised orbitals (lone pairs, bonds) make this
decomposition interpretable.

In [ ]:
def lmo_pair_overlap_matrix(mol1, C_lmo1, mol2, C_lmo2, level=3):
    """
    Returns omega[i, j] = 4 * int |phi_i^A|^2 |phi_j^B|^2 dr
    for all pairs of occupied LMOs from mol1 and mol2.
    """
    coords, weights = build_union_grid(mol1, mol2, level)

    ao1 = numint.eval_ao(mol1, coords)   # (npts, nAO1)
    ao2 = numint.eval_ao(mol2, coords)   # (npts, nAO2)

    n_occ1 = C_lmo1.shape[1]
    n_occ2 = C_lmo2.shape[1]

    # Evaluate each LMO on the grid: phi_i(r) = sum_mu C_mu,i * chi_mu(r)
    lmo1_grid = ao1 @ C_lmo1   # (npts, n_occ1)  phi_i^A(r)
    lmo2_grid = ao2 @ C_lmo2   # (npts, n_occ2)  phi_j^B(r)

    # Orbital densities: rho_i(r) = 2 |phi_i|^2
    orb_rho1 = 2 * lmo1_grid**2   # (npts, n_occ1)
    orb_rho2 = 2 * lmo2_grid**2   # (npts, n_occ2)

    # omega[i,j] = int rho_i^A(r) rho_j^B(r) dr
    #            = sum_g w_g * rho_i^A(r_g) * rho_j^B(r_g)
    #            = (orb_rho1 * w[:, None]).T @ orb_rho2
    omega_ij = (orb_rho1 * weights[:, None]).T @ orb_rho2   # (n_occ1, n_occ2)
    return omega_ij


omega_ij = lmo_pair_overlap_matrix(molA, C_boysA, molB, C_boysB)

print(f'Pair overlap matrix shape: {omega_ij.shape}  ({omega_ij.shape[0]} LMOs on A × {omega_ij.shape[1]} LMOs on B)')
print(f'Sum of all pairs Σ ω_ij  = {omega_ij.sum():.8f}')
print(f'Total Ω_AB (numerical)   = {omega_cmo:.8f}')
print(f'Difference               = {abs(omega_ij.sum() - omega_cmo):.2e}')

In [ ]:
# Assign simple labels: use LMO index + centroid distance from molecule A origin
lmo_labels_A = [f'LMO-A{i}' for i in range(omega_ij.shape[0])]
lmo_labels_B = [f'LMO-B{j}' for j in range(omega_ij.shape[1])]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(omega_ij, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label=r'$\omega_{ij}$ (pair overlap)')
ax.set_xticks(range(len(lmo_labels_B)))
ax.set_yticks(range(len(lmo_labels_A)))
ax.set_xticklabels(lmo_labels_B, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(lmo_labels_A, fontsize=8)
ax.set_xlabel('LMOs on molecule B')
ax.set_ylabel('LMOs on molecule A')
ax.set_title('LMO pair decomposition of $\\Omega_{AB}$ (Boys localization)')
plt.tight_layout()
plt.savefig('lmo_pair_overlap_heatmap.png', dpi=150)
plt.show()

# Print top contributing pairs
flat_idx = np.argsort(omega_ij.ravel())[::-1]
print('\nTop 5 orbital pair contributions:')
print(f'  {"LMO on A":>10}  {"LMO on B":>10}  {"ω_ij":>12}  {"% of total":>10}')
total = omega_ij.sum()
for k in flat_idx[:5]:
    i, j = divmod(k, omega_ij.shape[1])
    print(f'  {lmo_labels_A[i]:>10}  {lmo_labels_B[j]:>10}  {omega_ij[i,j]:>12.4e}  {100*omega_ij[i,j]/total:>9.1f}%')

### CMO pair decomposition (for comparison)

The same decomposition with canonical MOs gives the same total but each individual
$\omega_{ij}$ is less interpretable because CMOs are delocalized.

In [ ]:
C_occA = mfA.mo_coeff[:, mfA.mo_occ > 0]
C_occB = mfB.mo_coeff[:, mfB.mo_occ > 0]

omega_ij_cmo = lmo_pair_overlap_matrix(molA, C_occA, molB, C_occB)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(
    axes,
    [omega_ij_cmo, omega_ij],
    ['CMO pair decomposition', 'Boys LMO pair decomposition']
):
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, label=r'$\omega_{ij}$')
    ax.set_xlabel('MOs on molecule B')
    ax.set_ylabel('MOs on molecule A')
    ax.set_title(title)

plt.suptitle(
    f'Both sum to same Ω_AB = {omega_cmo:.4e}\n'
    f'CMO sum = {omega_ij_cmo.sum():.4e},  LMO sum = {omega_ij.sum():.4e}'
)
plt.tight_layout()
plt.savefig('cmo_vs_lmo_pair_decomposition.png', dpi=150)
plt.show()

## 9. Summary

| Topic | Key result |
|---|---|
| CMO density matrix | $\mathbf{P} = 2\mathbf{C}_{\text{occ}}\mathbf{C}_{\text{occ}}^T$ |
| LMO density matrix | Identical — $\mathbf{U}\mathbf{U}^T = \mathbf{I}$ cancels out |
| Total $\Omega_{AB}$ | Same for CMO and LMO (physical observable) |
| LMO pair $\omega_{ij}$ | Non-negative, sums to $\Omega_{AB}$, interpretable |
| CMO pair $\omega_{ij}$ | Same sum, but individual pairs are delocalized |

### Key takeaways

* **CMOs and LMOs give the same density matrix** and therefore the same $\Omega_{AB}$ — the
  total density is an orbital-representation-independent quantity.
* **LMOs are valuable for decomposition**: each $\omega_{ij}$ pinpoints which bond or lone pair
  on molecule A interacts spatially with which orbital on molecule B.
* The **Boys scheme** maximises spatial compactness; **Pipek–Mezey** preserves σ/π character
  — choice depends on what makes the decomposition most chemically intuitive.
* $\Omega_{AB}$ decays **exponentially** with intermolecular distance.
* $\Omega_{AB}$ underpins the **Hodgkin–Richards similarity index** $T_{HR} = 2\Omega_{AB}/(\Omega_{AA}+\Omega_{BB})$.